# TC3-45 – Analisar nulos, duplicidades e outliers

**Responsável:** Mauricio Cruz

**Sprint:** Fase 3

**Jira:** TC3-45

## Objetivo

Quantificar e investigar valores nulos, registros duplicados e possíveis outliers, registrando impactos e necessidades de tratamento.


In [17]:
import polars as pl
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

pl.Config.set_tbl_rows(-1)

polars.config.Config

### Carregar base de dados

In [18]:
# Ajuste o caminho conforme a localização real no seu ambiente

# Localiza a raiz do projeto a partir do diretório atual do notebook.
# Assim o notebook funciona no VS Code, Jupyter local ou outro clone do repositório.
def encontrar_raiz_projeto():
    atual = Path.cwd().resolve()
    for candidato in [atual, *atual.parents]:
        if (candidato / "dados" / "principal" / "ml_aluno.zip").exists():
            return candidato
    raise FileNotFoundError(
        "Não foi possível encontrar dados/principal/ml_aluno.zip. "
        "Execute o notebook dentro do repositório ou confira se os dados foram baixados."
    )

PROJECT_ROOT = encontrar_raiz_projeto()

BASE_PATH = PROJECT_ROOT / "dados" / "consolidado" / "base_alunos_municipios_por_uf"
# Leitura lazy (muito mais eficiente com partições)
df = pl.scan_parquet(
    f"{BASE_PATH}/sigla_uf=*/00000000.parquet",
    hive_partitioning=True
)
df_analise = df

In [19]:
df_analise = df_analise.collect()

print(f"Total de registros: {df_analise.height:,}")
print(f"Total de colunas  : {df_analise.width}")


Total de registros: 3,867,999
Total de colunas  : 49


In [20]:
# ============================================================
# 1. VALORES NULOS 
# ============================================================
print("\n" + "="*70)
print("1. VALORES NULOS – CONSOLIDAÇÃO")
print("="*70)

null_summary = (
    df_analise
    .null_count()
    .transpose(include_header=True, header_name="coluna", column_names=["nulos"])
    .with_columns(
        (pl.col("nulos") / df_analise.height * 100).round(2).alias("pct_nulos")
    )
    .sort("pct_nulos", descending=True)
)

print(null_summary.filter(pl.col("pct_nulos") > 0))

print(f"\nColunas com nulos     : {null_summary.filter(pl.col('nulos') > 0).height}")
print(f"Colunas 100% preenchidas: {null_summary.filter(pl.col('nulos') == 0).height}")


1. VALORES NULOS – CONSOLIDAÇÃO
shape: (33, 3)
┌─────────────────────────────────┬─────────┬───────────┐
│ coluna                          ┆ nulos   ┆ pct_nulos │
│ ---                             ┆ ---     ┆ ---       │
│ str                             ┆ u64     ┆ f64       │
╞═════════════════════════════════╪═════════╪═══════════╡
│ qtd_escolas_inse                ┆ 2121761 ┆ 54.85     │
│ qtd_alunos_inse                 ┆ 2121761 ┆ 54.85     │
│ media_inse                      ┆ 2121761 ┆ 54.85     │
│ pc_nivel_1                      ┆ 2121761 ┆ 54.85     │
│ pc_nivel_2                      ┆ 2121761 ┆ 54.85     │
│ pc_nivel_3                      ┆ 2121761 ┆ 54.85     │
│ pc_nivel_4                      ┆ 2121761 ┆ 54.85     │
│ pc_nivel_5                      ┆ 2121761 ┆ 54.85     │
│ pc_nivel_6                      ┆ 2121761 ┆ 54.85     │
│ pc_nivel_7                      ┆ 2121761 ┆ 54.85     │
│ pc_nivel_8                      ┆ 2121761 ┆ 54.85     │
│ nome uf               

In [21]:
# ============================================================
# 2. REGISTROS DUPLICADOS
# ============================================================
print("\n" + "="*70)
print("2. REGISTROS DUPLICADOS")
print("="*70)

# 2.1 Duplicatas de linha inteira
duplicatas_linha = df_analise.height - df_analise.unique().height
print(f"Duplicatas de linha inteira: {duplicatas_linha:,}")

# 2.2 Duplicatas pela chave de aluno (mais importante)
if "id_aluno" in df_analise.columns:
    dup_aluno = (
        df_analise
        .group_by("id_aluno")
        .agg(pl.len().alias("qtd"))
        .filter(pl.col("qtd") > 1)
    )
    print(f"Alunos com mais de um registro: {dup_aluno.height:,}")
    if dup_aluno.height > 0:
        print(dup_aluno.sort("qtd", descending=True).head(10))
else:
    print("Coluna 'id_aluno' não encontrada.")

# 2.3 Duplicatas por id_aluno + ano_referencia (se fizer sentido)
if all(c in df_analise.columns for c in ["id_aluno", "ano_referencia"]):
    dup_aluno_ano = (
        df_analise
        .group_by(["id_aluno", "ano_referencia"])
        .agg(pl.len().alias("qtd"))
        .filter(pl.col("qtd") > 1)
    )
    print(f"Duplicatas por (id_aluno + ano_referencia): {dup_aluno_ano.height:,}")
    if dup_aluno_ano.height > 0:
        print(dup_aluno_ano.sort("qtd", descending=True).head(5))


2. REGISTROS DUPLICADOS
Duplicatas de linha inteira: 0
Alunos com mais de um registro: 1,515,671
shape: (10, 2)
┌──────────┬─────┐
│ id_aluno ┆ qtd │
│ ---      ┆ --- │
│ str      ┆ u64 │
╞══════════╪═════╡
│ 24008133 ┆ 2   │
│ 15047407 ┆ 2   │
│ 52013692 ┆ 2   │
│ 26031280 ┆ 2   │
│ 43031412 ┆ 2   │
│ 33019356 ┆ 2   │
│ 15074536 ┆ 2   │
│ 26012893 ┆ 2   │
│ 33000188 ┆ 2   │
│ 43087783 ┆ 2   │
└──────────┴─────┘
Duplicatas por (id_aluno + ano_referencia): 0


In [22]:

# ============================================================
# 3. POSSÍVEIS OUTLIERS (método IQR)
# ============================================================
print("\n" + "="*70)
print("3. POSSÍVEIS OUTLIERS (Método IQR)")
print("="*70)

# Features numéricas relevantes para análise de outliers
features_outliers = [
    c for c in [
        "proficiencia", "peso_aluno",
        "valor_bolsa_familia", "valor_fundeb", "valor_receitas",
        "valor_pib", "valor_pib_per_capita",
        "media_inse",
        "taxa_alfabetizacao_municipio_ano_anterior",
        "media_portugues_municipio_ano_anterior",
        "qtd_alunos_inse", "qt_mat_bas", "qt_doc_bas", "qt_tur_bas"
    ] if c in df_analise.columns
]

outlier_results = []

for col in features_outliers:
    serie = df_analise[col].drop_nulls()
    if serie.len() < 100:
        continue

    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    limite_inf = q1 - 1.5 * iqr
    limite_sup = q3 + 1.5 * iqr

    n_outliers = serie.filter(
        (serie < limite_inf) | (serie > limite_sup)
    ).len()

    pct_outliers = round(n_outliers / serie.len() * 100, 2)

    outlier_results.append({
        "coluna": col,
        "q1": round(q1, 2),
        "q3": round(q3, 2),
        "iqr": round(iqr, 2),
        "limite_inf": round(limite_inf, 2),
        "limite_sup": round(limite_sup, 2),
        "n_outliers": n_outliers,
        "pct_outliers": pct_outliers,
        "min": round(serie.min(), 2),
        "max": round(serie.max(), 2)
    })

df_outliers = pl.DataFrame(outlier_results).sort("pct_outliers", descending=True)
print(df_outliers)


3. POSSÍVEIS OUTLIERS (Método IQR)
shape: (14, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ coluna    ┆ q1        ┆ q3        ┆ iqr       ┆ … ┆ n_outlier ┆ pct_outli ┆ min       ┆ max      │
│ ---       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ s         ┆ ers       ┆ ---       ┆ ---      │
│ str       ┆ f64       ┆ f64       ┆ f64       ┆   ┆ ---       ┆ ---       ┆ f64       ┆ f64      │
│           ┆           ┆           ┆           ┆   ┆ i64       ┆ f64       ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ valor_fun ┆ -1.0492e7 ┆ 7.0952e8  ┆ 7.2001e8  ┆ … ┆ 986371    ┆ 25.73     ┆ -4.5338e1 ┆ 3.5589e1 │
│ deb       ┆           ┆           ┆           ┆   ┆           ┆           ┆ 0         ┆ 0        │
│ valor_bol ┆ 2.1384139 ┆ 2.5300616 ┆ 2.3162202 ┆ … ┆ 576396    ┆ 14.9      ┆ 41280.0   ┆ 5.2370e9 │
│ sa_famili ┆ e7        ┆ 3e8       ┆ 4

## Resultados interpretados

### 1. Valores Nulos

- Já consolidado anteriormente.
- 33 colunas com nulos | 16 colunas 100% preenchidas.
- Bloco crítico (~54,85%) continua sendo o principal problema (INSE + Censo + PIB)


### 2. Registros Duplicados

| Tipo de duplicata | Quantidade | Interpretação |
|---|---:|---|
| Linha inteira | **0** | Nenhuma linha 100% idêntica. |
| `id_aluno` (qualquer ano) | **1.515.671** | Alunos que aparecem em mais de um ano (2023 e 2024). |
| `id_aluno` + `ano_referencia` | **0** | Não há duplicatas reais dentro do mesmo ano. |

Conclusão:

Não existem duplicatas problemáticas. A repetição de id_aluno é esperada porque a base contém dois anos. Cada aluno aparece no máximo uma vez por ano.
Impacto: Nenhum.

Tratamento necessário: Nenhum (manter como está).


### 3. Outliers (Método IQR) – Principais achados

| Feature | % Outliers | Min | Max | Observação importante | Tratamento recomendado |
|---|---:|---:|---:|---|---|
| `valor_fundeb` | **25,73%** | **-45,3 bi** | 35,6 bi | **Valores negativos** + extrema amplitude | Investigar a origem dos negativos + `log1p` ou tratamento especial. |
| `valor_bolsa_familia` | 14,90% | 41.280 | 5,24 bi | Forte assimetria | `log1p`. |
| `valor_receitas` | 14,39% | 8,4 mi | 257,5 bi | Forte assimetria | `log1p`. |
| `qt_mat_bas` / `qt_doc_bas` | 13,34% | — | — | Municípios grandes (capitais) | Avaliar se os valores são legítimos. |
| `valor_pib` | 13,18% | — | — | Mesmo padrão financeiro | `log1p`. |
| `qtd_alunos_inse` | 12,89% | 15 | 149.968 | Municípios grandes | Avaliar se os valores são legítimos. |
| `media_inse` | **0,00%** | 3,50 | 5,96 | Excelente comportamento | Nenhum tratamento necessário. |


Destaque crítico:

valor_fundeb possui valores negativos (mínimo ≈ -45 bilhões). Isso é semanticamente estranho para um indicador de transferência de recursos e precisa ser investigado na fonte.


### 4. Documentação Final do Requisito

#### Quantificação e Investigação de Valores Nulos, Duplicados e Outliers

#### 4.1 Valores Nulos
- **33 colunas** com valores nulos (67% das features).
- **16 colunas** 100% preenchidas.
- Bloco crítico (INSE + Censo Escolar + PIB + infraestrutura): **~54,85%** de nulos.
- Causa raiz: incompatibilidade temporal nos joins (bases externas de 2023 vs alunos de 2024).
- **Impacto:** Alto – perda de informação socioeconômica e de infraestrutura para mais da metade da base.
- **Tratamento necessário:** 
  1. Corrigir joins com fallback para o ano anterior (prioritário).
  2. Criar flags de missing.
  3. Imputação (mediana ou por UF) nas features restantes.

#### 4.2 Registros Duplicados
- Duplicatas de linha inteira: **0**.
- Duplicatas por (`id_aluno` + `ano_referencia`): **0**.
- Alunos presentes em mais de um ano: 1.515.671 (comportamento esperado).
- **Impacto:** Nenhum.
- **Tratamento necessário:** Nenhum.

#### 4.3 Possíveis Outliers (Método IQR)
- Features financeiras (`valor_fundeb`, `valor_bolsa_familia`, `valor_receitas`, `valor_pib`) apresentam entre **13% e 26%** de outliers.
- `valor_fundeb` possui valores **negativos** (mínimo ≈ -45,3 bilhões) → requer investigação.
- `media_inse` e features de desempenho educacional não apresentam outliers relevantes.
- Contagens escolares (`qt_mat_bas`, `qt_doc_bas`, `qtd_alunos_inse`) têm outliers legítimos (municípios grandes).

**Impacto:**  
Alto nas variáveis financeiras (podem distorcer modelos lineares e baseados em distância).  
Baixo nas features educacionais.

**Tratamentos recomendados:**
1. Investigar e tratar valores negativos de `valor_fundeb`.
2. Aplicar transformação `log1p` nas variáveis financeiras e em `peso_aluno`.
3. Avaliar winsorização leve após a transformação log (opcional).
4. Manter `media_inse` e `proficiencia` na escala original.

#### 4.4 Resumo de Prioridades de Tratamento

| Prioridade | Ação                                      | Justificativa                          |
|------------|-------------------------------------------|----------------------------------------|
| 1          | Corrigir joins (fallback de ano)          | Maior impacto na qualidade dos dados   |
| 2          | Investigar valores negativos de Fundeb    | Inconsistência semântica               |
| 3          | Transformação log1p (financeiras)         | Reduz assimetria e efeito de outliers  |
| 4          | Flags de missing + imputação              | Preservar informação e sinalizar falha |
| 5          | Nenhuma ação em duplicatas                | Não há duplicatas problemáticas        |




### Causa dos valores negativos em valor_fundeb

Os valores negativos não são erro de dado. Eles refletem a mecânica real do Fundeb.

#### Como o Fundeb funciona (resumo)

1. Cada município (e estado) contribui com 20% de uma cesta de impostos e transferências (ICMS, FPM, IPVA, etc.) para o fundo estadual do Fundeb.
2. Esses recursos são redistribuídos entre as redes de ensino do estado com base no número de matrículas (ponderadas).
3. A União ainda complementa o fundo (VAAF, VAAT e VAAR).

#### Por que aparece valor negativo?
O campo total_valor da base que você está usando representa, na prática, o resultado líquido do município no Fundeb:

Valor líquido ≈ (recursos recebidos do Fundeb) − (recursos que o município contribuiu para o fundo)

### Interpretação dos valores negativos em `valor_fundeb`

- Quando o município **recebe mais** do que contribuiu → valor **positivo**.
- Quando o município **contribui mais** do que recebe → valor **negativo** (perda líquida).

Esse comportamento é comum em municípios com:

- Alta arrecadação de impostos que compõem a cesta do Fundeb.
- Relativamente poucas matrículas na educação básica.
- Localização em estados com menor necessidade de redistribuição de recursos.

Essa **perda financeira líquida** é documentada na literatura e na contabilidade pública, representando a diferença negativa entre o valor retido na fonte e o montante que retornou ao município.


| Aspecto | Consequência |
|---|---|
| Natureza da variável | Não representa um “valor recebido puro”, mas sim um **saldo líquido**. |
| Valores negativos | São legítimos e informativos, indicando municípios que são contribuidores líquidos do Fundeb. |
| Uso em modelos | Pode ser mantido, pois o sinal carrega informação. Entretanto, a forte assimetria e a amplitude extrema (de −45 bilhões a +35 bilhões) exigem cuidado. |
| Tratamento recomendado | 1. Documentar que a variável representa um saldo líquido.<br>2. Avaliar transformações (ex.: logaritmo após *shift*) ou modelar separadamente valores positivos e negativos.<br>3. Criar uma feature binária `contribuidor_liquido` e outra com o valor absoluto do saldo. |


Os valores negativos de `valor_fundeb` não constituem inconsistências nos dados. Eles representam o saldo líquido do município no Fundeb, calculado pela diferença entre os recursos recebidos e a contribuição ao fundo.

Municípios com alta arrecadação e relativamente poucas matrículas na educação básica tendem a apresentar saldo negativo, caracterizando-se como **contribuidores líquidos do sistema de redistribuição de recursos**.
